# CyberTwin — Forensic Integrity & Tamper-Evident Chain of Custody
Synthetic academic implementation for M3–M6. **Never upload authentic personal or legal evidence.**

## Cell 1 — Environment Setup
Standard CPU runtime; Python standard library plus matplotlib.

In [ ]:
import hashlib, hmac, json, os, platform, random, ssl, time, tracemalloc
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
BASE = Path('/content/cybertwin')
EVID = BASE / 'evidence'
VAULT_PATH = BASE / 'evidence_vault.json'
CHUNK_SIZE = 64 * 1024
GENESIS = '0' * 64
EVID.mkdir(parents=True, exist_ok=True)
print(f'Python {platform.python_version()} | {ssl.OPENSSL_VERSION} | {platform.system()} {platform.machine()}')


## Cell 2 — Synthetic Evidence Generation

In [ ]:
CHAT = [
    '[2026-09-27 14:15:02] Impersonator: Hey, my phone broke. Using a temporary number.',
    '[2026-09-27 14:15:30] Impersonator: Urgent: Can you send Rs 50,000 to account #987654321?',
    '[2026-09-27 14:16:10] Victim: Who is this? Your display photo looks like Rohan.',
    '[2026-09-27 14:16:45] Impersonator: Yes it is Rohan! Please hurry, it is an emergency!',
]
SMS = ['27-09-26 14:18 IST: A/c XX4321 debited by Rs 50,000.00 via IMPS to A/c 987654321.', 'Not you? Report immediately on 1930 or cybercrime.gov.in']
original = ('\n'.join(CHAT) + '\n').encode('utf-8')
tampered = original.replace(b'#987654321', b'#987654329')
p_e1 = EVID/'E1_chat_log.txt'; p_e1_tampered = EVID/'E1_chat_log_TAMPERED.txt'; p_e2 = EVID/'E2_bank_sms.txt'; p_e3 = EVID/'E3_fake_profile.png'
p_e1.write_bytes(original); p_e1_tampered.write_bytes(tampered); p_e2.write_bytes(('\n'.join(SMS)+'\n').encode())
fig, ax = plt.subplots(figsize=(4,2)); ax.axis('off'); ax.text(.5,.5, "FAKE PROFILE: 'Rohan K.'\n(cloned display photo)", ha='center', va='center', weight='bold'); fig.savefig(p_e3,dpi=100,bbox_inches='tight'); plt.close(fig)
diffs=[(i,a,b,bin(a^b).count('1')) for i,(a,b) in enumerate(zip(original,tampered)) if a!=b]
print('Byte difference:', diffs[0], '| bytes changed:', len(diffs))


## Cell 3 — Streaming SHA-256 & NIST Validation

In [ ]:
def sha256_file(path, chunk_size=CHUNK_SIZE):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(chunk_size),b''): h.update(block)
    return h.hexdigest()
assert hashlib.sha256(b'abc').hexdigest() == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'
assert hashlib.sha256(b'abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq').hexdigest() == '248d6a61d20638b8e5c026930c3e6039a33ce45964ff2167f6ecedd419db06c1'
ref=sha256_file(p_e1)
for size in (1,7,64,65536): assert sha256_file(p_e1,size)==ref
print('NIST vectors: PASS'); print('Chunk-size invariance: PASS')
for p in sorted(EVID.iterdir()): print(p.name, sha256_file(p))


## Cell 4 — Evidence Vault & Hash-Chained Custody Ledger

In [ ]:
def utc_now(): return datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00','Z')
def entry_digest(entry):
    payload={k:v for k,v in entry.items() if k!='entry_hash'}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(',',':')).encode()).hexdigest()
class EvidenceVault:
    def __init__(self,path=VAULT_PATH,handler='Investigator-01 (Group G<no>)'):
        self.path=Path(path); self.handler=handler; self.records=[]; self.log=[]
        if self.path.exists(): self._load()
    def _load(self):
        d=json.loads(self.path.read_text()); self.records=d.get('records',[]); self.log=d.get('custody_log',[])
    def _save(self): self.path.write_text(json.dumps({'records':self.records,'custody_log':self.log},indent=2))
    def log_event(self,evidence_id,action,digest,result):
        prev=self.log[-1]['entry_hash'] if self.log else GENESIS
        e={'seq':len(self.log)+1,'timestamp_utc':utc_now(),'handler':self.handler,'evidence_id':evidence_id,'action':action,'sha256':digest,'result':result,'prev_hash':prev}; e['entry_hash']=entry_digest(e); self.log.append(e); self._save(); return e
    def ingest(self,path,case_label,description):
        path=Path(path); digest=sha256_file(path); eid=f'EV-{datetime.now(timezone.utc):%Y%m%d}-{len(self.records)+1:04d}'
        r={'evidence_id':eid,'case_label':case_label,'description':description,'original_filename':path.name,'file_size_bytes':path.stat().st_size,'timestamp_recorded_utc':utc_now(),'hash_algorithm':'SHA-256 (FIPS 180-4)','sha256_digest':digest}; self.records.append(r); self.log_event(eid,'INGESTED',digest,'BASELINE'); return r
    def get(self,eid): return next(r for r in self.records if r['evidence_id']==eid)
vault=EvidenceVault();
for p,desc in [(p_e1,'Synthetic chat'),(p_e2,'Synthetic bank SMS'),(p_e3,'Synthetic profile')]: vault.ingest(p,'CASE-2026-001',desc)
ANCHOR=(len(vault.log),vault.log[-1]['entry_hash']); print('Evidence records:',len(vault.records)); print('External anchor:',ANCHOR)


## Cell 5 — Integrity Audit: MATCH vs MISMATCH

In [ ]:
def verify_integrity(vault,path,eid):
    stored=vault.get(eid)['sha256_digest']; current=sha256_file(path); match=hmac.compare_digest(stored,current); result='MATCH' if match else 'MISMATCH'; vault.log_event(eid,'VERIFIED',current,result); return match
eid=vault.records[0]['evidence_id']
print('Authentic evidence:', '[MATCH / VERIFIED]' if verify_integrity(vault,p_e1,eid) else '[MISMATCH]')
print('Tampered evidence:', '[MATCH / VERIFIED]' if verify_integrity(vault,p_e1_tampered,eid) else '[MISMATCH / TAMPER DETECTED]')


## Cell 6 — Avalanche Analysis (1,000 trials)

In [ ]:
def bit_distance(a,b): return (int(a,16)^int(b,16)).bit_count()
baseline=sha256_file(p_e1); print('Single-bit output change:',bit_distance(baseline,sha256_file(p_e1_tampered)),'of 256 bits')
rng=random.Random(42); values=[]
for _ in range(1000):
    mutated=bytearray(original); pos=rng.randrange(len(mutated)*8); mutated[pos//8]^=1<<(pos%8); values.append(bit_distance(baseline,hashlib.sha256(mutated).hexdigest()))
mean=sum(values)/len(values); variance=sum((x-mean)**2 for x in values)/len(values); std=variance**.5
print(f'Mean bits changed: {mean:.2f} ({mean/256*100:.2f}%) | Std Dev: {std:.2f} | Min: {min(values)} | Max: {max(values)}')
plt.figure(figsize=(7,4.5)); plt.hist(values,bins=25,edgecolor='black'); plt.axvline(128,linestyle='--',label='Ideal: 128'); plt.title('SHA-256 Avalanche Effect — 1,000 Single-Bit Flips'); plt.xlabel('Output bits changed'); plt.ylabel('Frequency'); plt.legend(); plt.tight_layout(); plt.savefig(BASE/'fig_avalanche.png',dpi=200); plt.show()


## Cell 7 — Custody Chain Verification & Attack Simulation

In [ ]:
def verify_chain(v,anchor=None):
    issues=[]
    for i,e in enumerate(v.log):
        expected=GENESIS if i==0 else v.log[i-1]['entry_hash']
        if e.get('prev_hash')!=expected: issues.append(f"Seq {e.get('seq')}: Broken link")
        if entry_digest(e)!=e.get('entry_hash'): issues.append(f"Seq {e.get('seq')}: Log entry altered")
    for r in v.records:
        ing=[e for e in v.log if e.get('evidence_id')==r['evidence_id'] and e.get('action')=='INGESTED']
        if not ing: issues.append(f"{r['evidence_id']}: Missing baseline")
        elif r['sha256_digest']!=ing[0]['sha256']: issues.append(f"{r['evidence_id']}: Manifest digest differs from baseline")
    if anchor:
        seq,h=anchor; match=[e for e in v.log if e.get('seq')==seq]
        if not match: issues.append(f'Checkpoint seq {seq} missing')
        elif match[0]['entry_hash']!=h: issues.append(f'Checkpoint seq {seq} differs from anchored value')
    return issues
print('Genuine vault:', 'CHAIN INTACT (0 anomalies)' if not verify_chain(vault,ANCHOR) else 'ANOMALIES FOUND')
tampered_hash=sha256_file(p_e1_tampered)
for level in (1,2,3):
    data=json.loads(VAULT_PATH.read_text()); data['records'][0]['sha256_digest']=tampered_hash
    if level>=2: data['custody_log'][0]['sha256']=tampered_hash
    if level==3:
        prev=GENESIS
        for e in data['custody_log']: e['prev_hash']=prev; e['entry_hash']=entry_digest(e); prev=e['entry_hash']
    attack=BASE/f'vault_attack_L{level}.json'; attack.write_text(json.dumps(data,indent=2)); findings=verify_chain(EvidenceVault(attack),ANCHOR)
    print(f'Attack Level {level}:', 'DETECTED -> '+findings[0] if findings else 'NOT DETECTED')


## Cell 8 — Scalability & Memory Profiling

In [ ]:
for sz in (1,10,100):
    p=BASE/f'temp_{sz}mb.bin'; p.write_bytes(os.urandom(sz*1024*1024))
    tracemalloc.start(); t=time.perf_counter(); hs=sha256_file(p); elapsed=time.perf_counter()-t; _,peak=tracemalloc.get_traced_memory(); tracemalloc.stop()
    ha=hashlib.sha256(p.read_bytes()).hexdigest(); assert hs==ha
    print(f'{sz:>3} MB | {elapsed:.4f} s | streaming peak {peak/1024:.1f} KiB'); p.unlink()


## Cell 9 — Persistence Check

In [ ]:
reloaded=EvidenceVault(VAULT_PATH); assert reloaded.records==vault.records and reloaded.log==vault.log
print('Persistence Test (Full Rehydration from disk): PASS')
print(json.dumps(vault.records[0],indent=2))


## Cell 10 — Test Mapping
TC-01 NIST SHA-256 · TC-02 cross-tool/known digest · TC-03 chunk invariance · TC-04 authentic evidence · TC-05 one-bit tamper · TC-06 avalanche · TC-07 multi-type ingestion · TC-08 input validation · TC-09 custody tamper audit · TC-10 RAM scalability.

## Cell 11 — Optional Interactive Upload
For presentation only: upload harmless PNG/PDF/TXT-style demo files, ingest them, and run the same integrity check. Do not use authentic personal or legal evidence.